In [1]:
"""
Text Summarization with Hugging Face: BART and T5

Install:
    pip install transformers torch sentencepiece
"""

import torch
from transformers import (
    BartForConditionalGeneration,
    BartTokenizer,
    T5ForConditionalGeneration,
    T5Tokenizer,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# ---------------------------------------------------------------
# 1. BART  (facebook/bart-large-cnn is fine-tuned for news summarization)
# ---------------------------------------------------------------
def load_bart(model_name: str = "facebook/bart-large-cnn"):
    tokenizer = BartTokenizer.from_pretrained(model_name)
    model = BartForConditionalGeneration.from_pretrained(model_name).to(DEVICE)
    model.eval()
    return tokenizer, model


def summarize_bart(text, tokenizer, model, max_length=130, min_length=30, num_beams=4):
    inputs = tokenizer(
        text, return_tensors="pt", truncation=True, max_length=1024
    ).to(DEVICE)
    with torch.no_grad():
        ids = model.generate(
            **inputs,
            num_beams=num_beams,
            max_length=max_length,
            min_length=min_length,
            length_penalty=2.0,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )
    return tokenizer.decode(ids[0], skip_special_tokens=True)


# ---------------------------------------------------------------
# 2. T5  (uses the "summarize:" task prefix)
# ---------------------------------------------------------------
def load_t5(model_name: str = "t5-base"):
    tokenizer = T5Tokenizer.from_pretrained(model_name)
    model = T5ForConditionalGeneration.from_pretrained(model_name).to(DEVICE)
    model.eval()
    return tokenizer, model


def summarize_t5(text, tokenizer, model, max_length=130, min_length=30, num_beams=4):
    inputs = tokenizer(
        "summarize: " + text, return_tensors="pt", truncation=True, max_length=512
    ).to(DEVICE)
    with torch.no_grad():
        ids = model.generate(
            **inputs,
            num_beams=num_beams,
            max_length=max_length,
            min_length=min_length,
            length_penalty=2.0,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )
    return tokenizer.decode(ids[0], skip_special_tokens=True)


# ---------------------------------------------------------------
# 3. Long documents: split into chunks, summarize each, then join
# ---------------------------------------------------------------
def summarize_long(text, summarize_fn, tokenizer, model, chunk_words=400):
    words = text.split()
    chunks = [" ".join(words[i:i + chunk_words]) for i in range(0, len(words), chunk_words)]
    partial = [summarize_fn(c, tokenizer, model) for c in chunks]
    return " ".join(partial)


# ---------------------------------------------------------------
# Demo
# ---------------------------------------------------------------
if __name__ == "__main__":
    article = """
    Artificial intelligence has transformed many industries over the past decade.
    In healthcare, machine learning models now help doctors detect diseases such as
    cancer earlier and more accurately by analyzing medical images. In finance,
    algorithms detect fraudulent transactions in real time and automate trading.
    Meanwhile, natural language processing has made it possible for virtual assistants
    to understand speech, translate between languages, and summarize long documents.
    Despite these advances, researchers warn about challenges including bias in
    training data, lack of transparency in model decisions, and the environmental cost
    of training very large models. Governments around the world are now drafting
    regulations to ensure AI systems are developed and used responsibly.
    """

    print("=== BART ===")
    b_tok, b_model = load_bart()
    print(summarize_bart(article, b_tok, b_model), "\n")

    print("=== T5 ===")
    t_tok, t_model = load_t5()
    print(summarize_t5(article, t_tok, t_model), "\n")

    # Long document example (uses BART)
    # long_text = open("my_document.txt").read()
    # print(summarize_long(long_text, summarize_bart, b_tok, b_model))

=== BART ===


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

... (9 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

artificial intelligence has transformed many industries over the past decade . researchers warn about challenges including bias in training data, lack of transparency in model decisions . governments around the world are now drafting regulations to ensure AI systems are developed and used responsibly . 

